# Day C4 — 시간-주파수 분석: Morlet wavelet, STFT, 기저선 정규화

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/C-neural-signals/day-C4.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/C-neural-signals/day-C4.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q PyWavelets

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — wavelet 만들기와 진폭 되찾기

In [ ]:
import numpy as np

fs = 500
def morlet(freq, n_cycles, fs):
    """복소 Morlet wavelet: 복소 사인파 × 가우시안. 시간축은 0을 중심으로 대칭(홀수 개)"""
    wt = np.arange(-2, 2 + 1 / fs, 1 / fs)
    s = n_cycles / (2 * np.pi * freq)              # 가우시안의 표준편차(초)
    return wt, np.exp(2j * np.pi * freq * wt) * np.exp(-wt ** 2 / (2 * s ** 2)), s

for nc in [3, 7, 12]:
    wt, w, s = morlet(10, nc, fs)
    fwhm_t = 2 * np.sqrt(2 * np.log(2)) * s       # 시간 영역 반치폭
    fwhm_f = 2 * np.sqrt(2 * np.log(2)) / (2 * np.pi * s)   # 주파수 영역 반치폭
    print("10 Hz, %2d cycles: 시간 FWHM %4.0f ms | 주파수 FWHM %.1f Hz" % (nc, 1000 * fwhm_t, fwhm_f))

def wavelet_power(x, freqs, n_cycles, fs):
    """FFT 곱으로 wavelet convolution. 반환: (주파수 × 시간) 복소 결과"""
    wt, _, _ = morlet(freqs[0], 3, fs)
    L = len(x) + len(wt) - 1
    X = np.fft.fft(x, L)
    half = (len(wt) - 1) // 2
    out = np.zeros((len(freqs), len(x)), dtype=complex)
    for i, (f, nc) in enumerate(zip(freqs, n_cycles)):
        _, w, _ = morlet(f, nc, fs)
        W = np.fft.fft(w, L)
        W /= np.abs(W).max()                       # 주파수 영역 최댓값을 1로: 진폭이 원 단위로 나온다
        out[i] = np.fft.ifft(X * W)[half:half + len(x)]   # 날개 자르기
    return out

# 시험 신호: 1–2초에 진폭 1인 15 Hz, 2.5–3초에 진폭 2인 40 Hz
t = np.arange(0, 4, 1 / fs)
x = np.where((t >= 1) & (t < 2), np.sin(2 * np.pi * 15 * t), 0) \
  + np.where((t >= 2.5) & (t < 3), 2 * np.sin(2 * np.pi * 40 * t), 0)
freqs = np.logspace(np.log10(4), np.log10(60), 30)        # 4–60 Hz 로그 간격
n_cycles = np.logspace(np.log10(4), np.log10(10), 30)     # 낮은 주파수 4 cycles -> 높은 주파수 10 cycles
tf = wavelet_power(x, freqs, n_cycles, fs)
amp = 2 * np.abs(tf)    # 실수 사인파는 양·음 주파수에 반씩 나뉘므로 ×2 (C3과 같은 이유). power는 이것의 제곱

for f0, (a, b) in [(15, (1.3, 1.7)), (40, (2.6, 2.9))]:
    i = np.argmin(np.abs(freqs - f0))
    sel = (t >= a) & (t <= b)
    print("%.1f Hz 부근, %.1f–%.1f초 평균 진폭: %.2f" % (freqs[i], a, b, amp[i, sel].mean()))
i40 = np.argmin(np.abs(freqs - 40))
print("40 Hz 진폭이 최대인 시각: %.2f초" % t[np.argmax(amp[i40])])

### 예시 2 — total vs evoked, 그리고 세 가지 정규화

In [ ]:
import numpy as np

def morlet(freq, n_cycles, fs):
    wt = np.arange(-2, 2 + 1 / fs, 1 / fs)
    s = n_cycles / (2 * np.pi * freq)
    return np.exp(2j * np.pi * freq * wt) * np.exp(-wt ** 2 / (2 * s ** 2))

def tf_power(x, freqs, n_cycles, fs):
    """x: 시행 × 시간. 반환: 주파수 × 시행 × 시간의 power"""
    n_tr, n_t = x.shape
    M = len(morlet(freqs[0], 3, fs)); half = (M - 1) // 2
    L = n_t + M - 1
    X = np.fft.fft(x, L, axis=1)
    P = np.zeros((len(freqs), n_tr, n_t))
    for i, (f, nc) in enumerate(zip(freqs, n_cycles)):
        W = np.fft.fft(morlet(f, nc, fs), L); W /= np.abs(W).max()
        P[i] = np.abs(np.fft.ifft(X * W, axis=1)[:, half:half + n_t] * 2) ** 2
    return P

rng = np.random.default_rng(4)
fs = 250
t = np.arange(-1, 1.5, 1 / fs)
n_tr = 40
gate_a = ((t > 0.3) & (t < 0.9)).astype(float)      # alpha가 줄어드는 구간
gate_t = ((t > 0.1) & (t < 0.5)).astype(float)      # theta가 나타나는 구간
trials = np.zeros((n_tr, len(t)))
for k in range(n_tr):
    alpha = (1 - 0.6 * gate_a) * np.sin(2 * np.pi * 10 * t + rng.uniform(0, 2 * np.pi))  # 시행마다 위상이 다름
    theta = gate_t * np.sin(2 * np.pi * 6 * t)                                            # 매 시행 같은 위상
    trials[k] = alpha + theta + rng.normal(0, 1, len(t))

freqs = np.arange(4, 31, 1.0)
nc = np.linspace(4, 8, len(freqs))
total = tf_power(trials, freqs, nc, fs).mean(axis=1)            # 시행마다 power -> 평균 (total)
evoked = tf_power(trials.mean(axis=0, keepdims=True), freqs, nc, fs)[:, 0]   # 평균 파형(ERP)의 power

base = (t >= -0.7) & (t <= -0.2)                                 # 기저선: 자극 전 -0.7 ~ -0.2초
def normalize(P):
    b = P[:, base].mean(axis=1, keepdims=True)
    sd = P[:, base].std(axis=1, keepdims=True)
    return 10 * np.log10(P / b), 100 * (P - b) / b, (P - b) / sd

db, pct, z = normalize(total)
def at(A, f, a, b_):
    return A[np.argmin(np.abs(freqs - f)), (t >= a) & (t <= b_)].mean()

print("            | total 원power | dB     | % 변화 | z    | evoked 원power")
for name, f, a, b_ in [("alpha 10 Hz", 10, 0.4, 0.8), ("theta 6 Hz", 6, 0.2, 0.4)]:
    print("%-11s | %13.2f | %+6.2f | %+6.1f | %+5.1f | %.3f" % (name, at(total, f, a, b_), at(db, f, a, b_), at(pct, f, a, b_), at(z, f, a, b_), at(evoked, f, a, b_)))
print("기저선 구간 alpha total power %.2f vs evoked power %.3f" % (at(total, 10, -0.7, -0.2), at(evoked, 10, -0.7, -0.2)))

### 예시 3 — STFT의 창 길이, 그리고 PyWavelets

In [ ]:
import numpy as np
from scipy import signal
import pywt

fs = 500
t = np.arange(0, 4, 1 / fs)
# 2.000초에 아주 짧은(50 ms) 30 Hz 깜박임 + 내내 이어지는 10 Hz와 11 Hz
x = np.sin(2 * np.pi * 10 * t) + np.sin(2 * np.pi * 11 * t)
x += 3 * np.where((t >= 2.0) & (t < 2.05), np.sin(2 * np.pi * 30 * (t - 2.0)), 0)

for win in [0.1, 0.5, 2.0]:
    nper = int(win * fs)
    f, tt, S = signal.spectrogram(x, fs, window="hann", nperseg=nper, noverlap=nper - nper // 10)
    # (a) 10 Hz와 11 Hz가 따로 보이는가? 9–12 Hz 사이 봉우리가 2개면 구분된 것
    sel = (f >= 9) & (f <= 12)
    spec = S[sel].mean(axis=1)
    n_pk = len(signal.find_peaks(spec)[0])
    # (b) 30 Hz 깜박임이 얼마나 넓게 번지는가: 30 Hz 행에서 최고값 절반 이상인 시간 폭
    row = S[np.argmin(np.abs(f - 30))]
    width = (row > row.max() / 2).sum() * (tt[1] - tt[0])
    print("창 %.1f초: 주파수 간격 %.2f Hz | 10/11 Hz 구분: %s | 30 Hz 번짐 폭 %.2f초" % (win, f[1], "예" if n_pk >= 2 else "아니오", width))

# PyWavelets의 연속 wavelet 변환(복소 Morlet 'cmor'): scale을 주파수로 바꾸어 쓴다
wav = "cmor1.5-1.0"
target = np.array([10.5, 30.0])
scales = pywt.frequency2scale(wav, target / fs)
coef, fr = pywt.cwt(x, scales, wav, sampling_period=1 / fs)
print("pywt scale -> 주파수 확인:", np.round(fr, 2), "Hz")
print("30 Hz 계수가 가장 큰 시각: %.3f초" % t[np.argmax(np.abs(coef[1]))])